# Stage 1 - Step 2: Feature extraction (the part we only do once)

This is arguably the most important notebook in the whole project: everything else
(linear probe, prototypes, plots) reads from files this notebook produces and never
touches the encoders again. If we mess this up, everything downstream is wrong, so
we're being extra careful with the preprocessing here.

What we're extracting:
- **ResNet-18** (ImageNet-pretrained, frozen) -> on **both** DTD and Aircraft, 512-dim features
- **DINOv2 ViT-S/14** (frozen) -> on **Aircraft only** (our choice - see the reasoning in
  the writeup, basically we wanted the ResNet-18 vs DINOv2 comparison on the harder,
  fine-grained dataset)

For each (encoder, dataset) pair we extract features for **all three splits**
(train/val/test) and save them as plain `.pt` files. K-shot subsampling (5-shot,
10-shot) happens later, in the linear probe / prototype notebooks - not here. Here we
just cache the full thing once.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os

PROJECT_ROOT = '/content/drive/MyDrive/cvlab_stage1'
DATA_ROOT = os.path.join(PROJECT_ROOT, 'data')
CACHE_ROOT = os.path.join(PROJECT_ROOT, 'features')
RESULTS_ROOT = os.path.join(PROJECT_ROOT, 'results')

for p in [DATA_ROOT, CACHE_ROOT, RESULTS_ROOT]:
    os.makedirs(p, exist_ok=True)

print('Using project root:', PROJECT_ROOT)

In [ ]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
import time

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print('device:', device)
if device == 'cpu':
    print('WARNING: no GPU detected, this will be much slower. Runtime > Change runtime type > GPU')

## Generic feature extraction helper

Same function works for any frozen encoder - we just pass in the model and it runs
a forward pass over the whole dataset in batches, no gradients, and stacks everything
into one big tensor. Splitting this into its own function means we don't repeat the
same loop 3 times with copy-paste bugs.

In [ ]:
@torch.no_grad()
def extract_features(model, dataset, transform, batch_size=128, num_workers=2):
    """Runs a frozen model over a whole dataset and returns (features, labels) tensors."""
    model.eval()
    model.to(device)

    # wrap the dataset so it applies our transform (some torchvision datasets don't
    # let you set .transform after creation cleanly, so we do it with a tiny wrapper)
    class _Wrapped(torch.utils.data.Dataset):
        def __init__(self, base, tf):
            self.base = base
            self.tf = tf
        def __len__(self):
            return len(self.base)
        def __getitem__(self, idx):
            img, label = self.base[idx]
            return self.tf(img), label

    loader = DataLoader(_Wrapped(dataset, transform), batch_size=batch_size,
                         shuffle=False, num_workers=num_workers, pin_memory=(device=='cuda'))

    all_feats, all_labels = [], []
    t0 = time.time()
    for i, (imgs, labels) in enumerate(loader):
        imgs = imgs.to(device, non_blocking=True)
        feats = model(imgs)
        all_feats.append(feats.cpu())
        all_labels.append(labels)
        if i % 5 == 0:
            print(f'  batch {i+1}/{len(loader)}', end='\r', flush=True)
    print(f'  done, {len(loader)} batches in {time.time()-t0:.1f}s' + ' ' * 10)

    return torch.cat(all_feats, dim=0), torch.cat(all_labels, dim=0)

## Setting up ResNet-18 (frozen)

We load the standard torchvision ImageNet-pretrained weights, then replace the final
`fc` layer with `nn.Identity()`. That's a quick trick that turns the classifier head
into a no-op, so calling the model just gives us the 512-dim feature vector that used
to feed into `fc` - exactly what the assignment asks for ("extract the 512-dimensional
representation before the final classification layer").

We use `weights.transforms()` for preprocessing instead of writing our own resize/crop/
normalize by hand - this guarantees we're using the *exact* preprocessing this
checkpoint was trained with, which the assignment specifically asks for.

In [ ]:
from torchvision.models import resnet18, ResNet18_Weights

resnet_weights = ResNet18_Weights.IMAGENET1K_V1
resnet = resnet18(weights=resnet_weights)
resnet.fc = nn.Identity()   # now the model outputs 512-dim features instead of 1000-class logits

for p in resnet.parameters():
    p.requires_grad = False  # frozen, no training happens here or anywhere downstream

resnet_transform = resnet_weights.transforms()
print(resnet_transform)

## Setting up DINOv2 ViT-S/14 (frozen)

Loaded straight from Meta's repo via `torch.hub` - no separate pip package needed.
Calling the model on a batch of images returns the class-token embedding by default
(384-dim for the small variant), which is what we want.

The preprocessing here follows the standard DINOv2 eval recipe: resize shorter side
to 256, center crop to 224, normalize with ImageNet statistics. (DINOv2 was trained
with these same statistics even though it's not supervised on ImageNet labels - it's
just the standard normalization most vision backbones use.)

In [ ]:
dinov2 = torch.hub.load('facebookresearch/dinov2', 'dinov2_vits14')

for p in dinov2.parameters():
    p.requires_grad = False

from torchvision import transforms

dinov2_transform = transforms.Compose([
    transforms.Resize(256, interpolation=transforms.InterpolationMode.BICUBIC),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(mean=(0.485, 0.456, 0.406), std=(0.229, 0.224, 0.225)),
])
print(dinov2_transform)
print('DINOv2 output dim check:', dinov2(torch.randn(1, 3, 224, 224)).shape)

## Copying data from Drive to local disk first

Important gotcha: Drive is mounted over the network (technically a FUSE filesystem),
which is fine for reading/writing a handful of big files but genuinely slow for
reading thousands of individual small image files one at a time - which is exactly
what a plain folder copy or `DataLoader` does. Copying the already-*extracted*
folders (thousands of loose .jpg files) would be just as slow as reading them
directly, because it's still one file at a time.

The fix: torchvision never deleted the original downloaded `.tar.gz` archive files
after extracting them (default behavior, `remove_finished=False`) - they're still
sitting on Drive next to the extracted images. So instead of copying thousands of
small files, we copy just the **2 archive files** (one continuous read each, fast
even over Drive) to local disk, then extract them locally with Python's `tarfile`
module. All the actual image reading during feature extraction then happens on local
disk, which is fast.

This cell searches for any `.tar.gz` under our Drive data folder, copies each one
locally (preserving the same relative folder layout torchvision expects), and
extracts it. It should take well under a minute - if it's still running after a
few minutes, stop and let me know, something else is going on.

In [ ]:
import glob, shutil, tarfile

LOCAL_DATA_ROOT = '/content/data'
os.makedirs(LOCAL_DATA_ROOT, exist_ok=True)

archives = glob.glob(os.path.join(DATA_ROOT, '**', '*.tar.gz'), recursive=True)
print(f'found {len(archives)} archive(s) on Drive:')
for a in archives:
    print(' -', a)
assert len(archives) >= 2, 'expected at least 2 archives (dtd + aircraft) - did notebook 1 finish downloading?'

for archive_path in archives:
    rel_dir = os.path.relpath(os.path.dirname(archive_path), DATA_ROOT)
    local_dir = os.path.join(LOCAL_DATA_ROOT, rel_dir)
    os.makedirs(local_dir, exist_ok=True)
    local_archive = os.path.join(local_dir, os.path.basename(archive_path))

    if not os.path.exists(local_archive):
        print(f'copying {os.path.basename(archive_path)} ({os.path.getsize(archive_path)/1e6:.0f} MB) to local disk...')
        shutil.copyfile(archive_path, local_archive)
    else:
        print(f'{os.path.basename(archive_path)} already copied locally, skipping copy')

    # only extract if this folder doesn't already have extracted content in it
    already_extracted = any(not f.endswith('.tar.gz') for f in os.listdir(local_dir))
    if not already_extracted:
        print(f'  extracting {os.path.basename(archive_path)} locally...')
        with tarfile.open(local_archive) as tar:
            tar.extractall(local_dir)
    print(f'  ready: {local_dir}')

print()
print('local data copy/extract done.')

## Loading the datasets (now from local disk, not Drive)

In [ ]:
from torchvision.datasets import DTD, FGVCAircraft

dtd_root = os.path.join(LOCAL_DATA_ROOT, 'dtd')
dtd_train = DTD(root=dtd_root, split='train', partition=1)
dtd_val   = DTD(root=dtd_root, split='val',   partition=1)
dtd_test  = DTD(root=dtd_root, split='test',  partition=1)

aircraft_root = os.path.join(LOCAL_DATA_ROOT, 'fgvc_aircraft')
aircraft_train = FGVCAircraft(root=aircraft_root, split='train', annotation_level='variant')
aircraft_val   = FGVCAircraft(root=aircraft_root, split='val',   annotation_level='variant')
aircraft_test  = FGVCAircraft(root=aircraft_root, split='test',  annotation_level='variant')

print('DTD:', len(dtd_train), len(dtd_val), len(dtd_test))
print('Aircraft:', len(aircraft_train), len(aircraft_val), len(aircraft_test))

## Extract + cache: ResNet-18 on DTD

This is the first of our three (encoder, dataset) combinations. Should take under a
minute or two on a Colab GPU for a dataset this size.

In [ ]:
splits = {'train': dtd_train, 'val': dtd_val, 'test': dtd_test}

for split_name, ds in splits.items():
    print(f'ResNet-18 / DTD / {split_name}')
    feats, labels = extract_features(resnet, ds, resnet_transform)
    save_path = os.path.join(CACHE_ROOT, f'resnet18_dtd_{split_name}.pt')
    torch.save({'features': feats, 'labels': labels, 'classes': dtd_train.classes}, save_path)
    print(f'  saved {feats.shape} -> {save_path}')

## Extract + cache: ResNet-18 on Aircraft

In [ ]:
splits = {'train': aircraft_train, 'val': aircraft_val, 'test': aircraft_test}

for split_name, ds in splits.items():
    print(f'ResNet-18 / Aircraft / {split_name}')
    feats, labels = extract_features(resnet, ds, resnet_transform)
    save_path = os.path.join(CACHE_ROOT, f'resnet18_aircraft_{split_name}.pt')
    torch.save({'features': feats, 'labels': labels, 'classes': aircraft_train.classes}, save_path)
    print(f'  saved {feats.shape} -> {save_path}')

## Extract + cache: DINOv2 on Aircraft

Same loop, different model and transform. This one's a bigger model so expect it to
take a bit longer than the ResNet-18 passes.

In [ ]:
splits = {'train': aircraft_train, 'val': aircraft_val, 'test': aircraft_test}

for split_name, ds in splits.items():
    print(f'DINOv2 / Aircraft / {split_name}')
    feats, labels = extract_features(dinov2, ds, dinov2_transform)
    save_path = os.path.join(CACHE_ROOT, f'dinov2_aircraft_{split_name}.pt')
    torch.save({'features': feats, 'labels': labels, 'classes': aircraft_train.classes}, save_path)
    print(f'  saved {feats.shape} -> {save_path}')

## Sanity check: reload everything and check shapes

Quick check that nothing got saved wrong - reload each file fresh from disk and print
shapes. Expected feature dims: 512 for ResNet-18, 384 for DINOv2.

In [ ]:
expected = [
    ('resnet18_dtd_train.pt', 512), ('resnet18_dtd_val.pt', 512), ('resnet18_dtd_test.pt', 512),
    ('resnet18_aircraft_train.pt', 512), ('resnet18_aircraft_val.pt', 512), ('resnet18_aircraft_test.pt', 512),
    ('dinov2_aircraft_train.pt', 384), ('dinov2_aircraft_val.pt', 384), ('dinov2_aircraft_test.pt', 384),
]

for fname, expected_dim in expected:
    path = os.path.join(CACHE_ROOT, fname)
    d = torch.load(path)
    feat_dim = d['features'].shape[1]
    n = d['features'].shape[0]
    status = 'OK' if feat_dim == expected_dim else 'MISMATCH'
    print(f'{fname:35s} n={n:5d}  dim={feat_dim:4d}  ({status})')
    assert feat_dim == expected_dim, f'{fname} has wrong feature dim'

print()
print('All 9 cached feature files look correct.')

## Done with this notebook once all 9 files check out

We now have every feature we'll ever need cached in `CACHE_ROOT` on Drive:
- `resnet18_dtd_{train,val,test}.pt`
- `resnet18_aircraft_{train,val,test}.pt`
- `dinov2_aircraft_{train,val,test}.pt`

From here on, **nothing touches the encoders again**. Notebook 3 (linear probe) and
notebook 4 (prototypes) both just load these `.pt` files and work with plain tensors,
which is why they'll run in seconds/minutes instead of needing GPU time for the
encoder itself.